In [3]:
import nltk
import random
import string
import warnings

from nltk.corpus import stopwords
from nltk.classify.scikitlearn import SklearnClassifier

from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression, SGDClassifier

warnings.simplefilter(action='ignore', category=FutureWarning)

nltk.download('stopwords')
nltk.download('punkt')

def find_feature(word_features, message):
    message = message.lower()
    return {word: (word in message) for word in word_features}

def create_training_testing():
    with open('SMSSpamCollection.txt', encoding='utf-8') as f:
        messages = f.read().split('\n')

    all_messages = []
    all_words = []

    stop_words = set(stopwords.words('english'))

    for message in messages:
        if len(message.split('\t')) != 2:
            continue
        
        label, text = message.split('\t')
        category = "spam" if label == "spam" else "ham"
        all_messages.append((text, category))

        
        text = text.translate(str.maketrans('', '', string.punctuation))

        for word in text.split():
            if word.lower() not in stop_words:
                all_words.append(word.lower())

    random.shuffle(all_messages)

    all_words = nltk.FreqDist(all_words)
    word_features = list(all_words.keys())[:2000]

    featureset = [(find_feature(word_features, msg), category) for (msg, category) in all_messages]

    split = int(len(featureset) * 0.75)
    trainingset = featureset[:split]
    testingset = featureset[split:]

    print("Total:", len(featureset))
    print("Training:", len(trainingset))
    print("Testing:", len(testingset))

    return word_features, trainingset, testingset

def train_models(trainingset, testingset):
    results = {}

    
    nb = nltk.NaiveBayesClassifier.train(trainingset)
    acc_nb = nltk.classify.accuracy(nb, testingset) * 100
    print("\nNaive Bayes Accuracy:", acc_nb)
    results["Naive Bayes"] = (nb, acc_nb)

    
    mnb = SklearnClassifier(MultinomialNB())
    mnb.train(trainingset)
    acc_mnb = nltk.classify.accuracy(mnb, testingset) * 100
    print("MultinomialNB Accuracy:", acc_mnb)
    results["MultinomialNB"] = (mnb, acc_mnb)

    
    bnb = SklearnClassifier(BernoulliNB())
    bnb.train(trainingset)
    acc_bnb = nltk.classify.accuracy(bnb, testingset) * 100
    print("BernoulliNB Accuracy:", acc_bnb)
    results["BernoulliNB"] = (bnb, acc_bnb)

    
    lr = SklearnClassifier(LogisticRegression(max_iter=200))
    lr.train(trainingset)
    acc_lr = nltk.classify.accuracy(lr, testingset) * 100
    print("Logistic Regression Accuracy:", acc_lr)
    results["Logistic Regression"] = (lr, acc_lr)

    
    sgd = SklearnClassifier(SGDClassifier())
    sgd.train(trainingset)
    acc_sgd = nltk.classify.accuracy(sgd, testingset) * 100
    print("SGD Accuracy:", acc_sgd)
    results["SGD"] = (sgd, acc_sgd)

    return results

word_features, trainingset, testingset = create_training_testing()
models = train_models(trainingset, testingset)

mails = [
    "Ankur is husband of Sahana",
    "Nature is beautiful",
    "FREE money win cash now!!!",
    "Let's meet tomorrow for lunch"
]

print("\n\nPREDICTIONS")
print("="*40)

for name, (model, acc) in models.items():
    print(f"\n{name} ({acc:.2f}%)")
    print("-"*30)
    for mail in mails:
        features = find_feature(word_features, mail)
        print(f"{mail} -> {model.classify(features)}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Total: 5574
Training: 4180
Testing: 1394

Naive Bayes Accuracy: 98.56527977044476
MultinomialNB Accuracy: 98.63701578192251
BernoulliNB Accuracy: 98.35007173601149
Logistic Regression Accuracy: 98.99569583931134
SGD Accuracy: 98.85222381635582


PREDICTIONS

Naive Bayes (98.57%)
------------------------------
Ankur is husband of Sahana -> ham
Nature is beautiful -> ham
FREE money win cash now!!! -> ham
Let's meet tomorrow for lunch -> ham

MultinomialNB (98.64%)
------------------------------
Ankur is husband of Sahana -> ham
Nature is beautiful -> ham
FREE money win cash now!!! -> spam
Let's meet tomorrow for lunch -> ham

BernoulliNB (98.35%)
------------------------------
Ankur is husband of Sahana -> ham
Nature is beautiful -> ham
FREE money win cash now!!! -> ham
Let's meet tomorrow for lunch -> ham

Logistic Regression (99.00%)
------------------------------
Ankur is husband of Sahana -> ham
Nature is beautiful -> ham
FREE money win cash now!!! -> ham
Let's meet tomorrow for lunc